## 📖 Dicionário de Dados

| **Coluna** | **Descrição** |
|---|---|
| `DATA` | Data da compra |
| `CO_ID` | Identificação da compra (número da nota fiscal) |
| `CL_ID` | Identificação do cliente (número do cliente) |
| `CL_GENERO` | Sexo biológico informado pelo cliente |
| `CL_EC` | Estado civil do cliente |
| `CL_FHL` | Número de filhos do cliente |
| `CL_SEG` | Segmentação econômica do cliente (Classe A, B ou C) |
| `PR_ID` | Código do produto adquirido (SKU) |
| `PR_CAT` | Categoria do produto adquirido |
| `PR_NOME` | Nome do produto adquirido |

### Valores de `CL_EC` (Estado Civil)

| **Código** | **Descrição** |
|---|---|
| `1` | Casado ou união estável |
| `2` | Divorciado |
| `3` | Separado |
| `4` | Solteiro |
| `5` | Viúvo |

## 🔍 Design dos dados importados

| **Critério** | **DATA** | **CO_ID** | **CL_ID** | **CL_GENERO** | **CL_EC** | **CL_FHL** | **CL_SEG** | **PR_ID** | **PR_CAT** | **PR_NOME** |
|---|---|---|---|---|---|---|---|---|---|---|
| Tipo de dado atual | str | int64 | int64 | str | int64 | int64 | str | int64 | str | str |
| Tipo de dado após tratamento | datetime64 | int64 | int64 | string | string | int64 | string | int64 | string | string |
| Duplicatas | | | | | | | | | | |
| Valores nulos | | | | | | | | | | |
| Impacto no negócio | | | | | | | | | | |

## ETAPA DE IMPORTAÇÃO DE BIBLIOTECAS QUE SERÃO USADAS

In [132]:
import sys
import numpy as np
import pandas as pd
import importlib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

BASE_DIR = Path().resolve().parent
sys.path.append(str(BASE_DIR))

import utils.data_utils as du
importlib.reload(du)


<module 'utils.data_utils' from '/home/calliari/Documents/Estudos/MiniProjetoSCTEC/utils/data_utils.py'>

## IMPORTANDO OS DADOS DO CSV

In [133]:
caminho = "../data/base_varejo.csv"
df = pd.read_csv (caminho,sep=";")

## Visualizações iniciais do DF criado ##

In [134]:
# Primeiras 5 linhas , para confirmar a leitura do arquivo
print (df.head(5))

         DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID     PR_CAT  \
0  01/02/2019   1000    534         M      4       1      C     67    BEBIDAS   
1  01/02/2019   1000    534         M      4       1      C     70    BEBIDAS   
2  01/02/2019   1000    534         M      4       1      C    178    HIGIENE   
3  01/02/2019   1000    534         M      4       1      C      4  ALIMENTOS   
4  01/02/2019   1000    534         M      4       1      C    175    LIMPEZA   

                PR_NOME  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13  
0  REFRIGERANTE GUARANA          NaN          NaN          NaN          NaN  
1   REFRIGERANTE OUTROS          NaN          NaN          NaN          NaN  
2       LENCO UMEDECIDO          NaN          NaN          NaN          NaN  
3               ABACAXI          NaN          NaN          NaN          NaN  
4     LIMPADOR MULTIUSO          NaN          NaN          NaN          NaN  


In [135]:
# Tipo dos dados encontrados
print (df.dtypes)

DATA               str
CO_ID            int64
CL_ID            int64
CL_GENERO          str
CL_EC            int64
CL_FHL           int64
CL_SEG             str
PR_ID            int64
PR_CAT             str
PR_NOME            str
Unnamed: 10    float64
Unnamed: 11    float64
Unnamed: 12    float64
Unnamed: 13    float64
dtype: object


In [136]:
# Linhas X Colunas
print (f"Quantidade de Linhas e Colunas localizadas")
print (f"Linhas: {df.shape[0]} \nColunas: {df.shape[1]}")

Quantidade de Linhas e Colunas localizadas
Linhas: 830000 
Colunas: 14


In [137]:
print ("\nVisualização das estatísticas dos dados (Quartis/Media/Minimo/Max)\n")
print (df.describe())


Visualização das estatísticas dos dados (Quartis/Media/Minimo/Max)

          CO_ID     CL_ID     CL_EC    CL_FHL     PR_ID  Unnamed: 10  \
count 830000.00 830000.00 830000.00 830000.00 830000.00         0.00   
mean  460045.09    499.60      2.60      1.15    115.05          NaN   
std   265465.25    287.57      1.17      1.42     66.13          NaN   
min     1000.00      1.00      1.00      0.00      1.00          NaN   
25%   233117.00    254.00      2.00      0.00     58.00          NaN   
50%   456517.00    498.00      3.00      0.00    115.00          NaN   
75%   690132.00    746.00      4.00      2.00    172.00          NaN   
max   919822.00   1000.00      5.00      4.00    229.00          NaN   

       Unnamed: 11  Unnamed: 12  Unnamed: 13  
count         0.00         0.00         0.00  
mean           NaN          NaN          NaN  
std            NaN          NaN          NaN  
min            NaN          NaN          NaN  
25%            NaN          NaN          NaN  


In [138]:
print ("\nSomatória de valores nulos por Coluna, aqui verificamos a existência de 4 colunas não nomeadas que estão todas nulas")
print (df.isnull().sum())


Somatória de valores nulos por Coluna, aqui verificamos a existência de 4 colunas não nomeadas que estão todas nulas
DATA                0
CO_ID               0
CL_ID               0
CL_GENERO           0
CL_EC               0
CL_FHL              0
CL_SEG              0
PR_ID               0
PR_CAT              0
PR_NOME             0
Unnamed: 10    830000
Unnamed: 11    830000
Unnamed: 12    830000
Unnamed: 13    830000
dtype: int64


## Realizado uma primeira inspeção no documento importado, inicia agora os tratamentos dos dados

* Remoção das Colunas ( Unnamed: 10 / Unnamed: 11 / Unnamed: 12 / Unnamed: 13) 

In [139]:
# DROP Colunas zeradas
df = df.drop(columns=['Unnamed: 10', 'Unnamed: 11','Unnamed: 12','Unnamed: 13'])
# Confirmação que as colunas foram excluídas:
print (df.shape)
# Antes da remoção, eram 830000 x 14, confirmando assim a exclusão

(830000, 10)


In [140]:
# Substituição de String Vazias e Nulls utilizando o np.nan para obter NaN real

df = df.replace(['NULL', 'N/A', '', '#N/D',  ' '], np.nan, inplace=True)
print(df.isnull().sum())

DATA            0
CO_ID           0
CL_ID           0
CL_GENERO       0
CL_EC           0
CL_FHL          0
CL_SEG          0
PR_ID           0
PR_CAT       3650
PR_NOME      3650
dtype: int64


In [ ]:
# Padronizar Gênero F/M por FEMINIMO/MASCULINO
print (f"Antes da padronizaçao") 
print (df['CL_GENERO'].head(5))

df['CL_GENERO'] = du.padroniza_genero(df)
print (f"\nApós padronizaçao") 
print (df['CL_GENERO'].head(5))

Antes da padronizaçao
0    M
1    M
2    M
3    M
4    M
Name: CL_GENERO, dtype: str

Após padronizaçao
0    MASCULINO
1    MASCULINO
2    MASCULINO
3    MASCULINO
4    MASCULINO
Name: CL_GENERO, dtype: str


In [ ]:
# Padronizar Estado Civil de INT (1/2/3/4/5) para o correspodente em STR (CASADO OU UNIÃO ESTÁVEL, DIVORCIADO, SEPARADO, SOLTEIRO, VIÚVO)

print (f"Antes da padronizaçao")
print (df['CL_EC'].head(5)) 

df['CL_EC'] = du.padroniza_estado_civil(df)
print (f"\nApós padronizaçao") 
print (df['CL_EC'].head(5)) 

Antes da padronizaçao
0    4
1    4
2    4
3    4
4    4
Name: CL_EC, dtype: int64

Após padronizaçao
0    SOLTEIRO
1    SOLTEIRO
2    SOLTEIRO
3    SOLTEIRO
4    SOLTEIRO
Name: CL_EC, dtype: str
